# Naive Bayes for mixed features

Income is the field people decline to answer. Age, education, job sector, hours worked — those they
will usually give you. So the question worth money is whether the fields you have can tell you the
one you do not.

This notebook's dataset puts exactly that question, over 1994 US census records: did this person
earn more than \$50K? Fourteen fields describe each one, and they come in two sorts — labels like a
job sector, numbers like an age. Your classifier has to read both.

The work is four gaps in **`naive_bayes.py`**, and that module is the only graded artefact. The
sections run in the order the work should be done: look at the data first, let it tell you what
kinds of column there are, and write the model down once you know what it has to cope with.

In [ ]:
# On Colab, fetch the helper modules and the template that sit next to this notebook in the repo.
import pathlib
import urllib.request

BASE_URL = (
    "https://raw.githubusercontent.com/girafe-ai/ml-course/homeworks/contest/classic_naive_bayes"
)

try:
    import ucimlrepo
except ImportError:
    %pip install --quiet ucimlrepo

for module in ("mixed_nb_data.py", "mixed_nb_eval.py", "naive_bayes.py"):
    if not pathlib.Path(module).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{module}", module)
        print(f"downloaded {module}")

In [ ]:
import importlib
import time

import matplotlib.pyplot as plt
import mixed_nb_data
import mixed_nb_eval
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

## 1. The dataset

[Adult](https://archive.ics.uci.edu/dataset/2/adult) is that census extract: some 49 000
respondents, fourteen fields each, and the \$50K label. `ucimlrepo` fetches it and — more usefully
for what follows — hands over UCI's own *variable metadata*, a small table naming every column,
saying whether it is a feature or the target, and declaring its type.

Keep that table in view. Section 4 leans on it, and section 2 is partly about where it disagrees
with what the values look like.

In [ ]:
adult = fetch_ucirepo(id=2)

TARGET = str(adult.data.targets.columns[0])
data = pd.concat([adult.data.features, adult.data.targets], axis=1)
variables = adult.variables.set_index("name")

print(f"{len(data)} rows, {data.shape[1] - 1} features, target {TARGET!r}")
display(variables.loc[list(data.columns), ["role", "type", "description"]])
display(data.head())

## 2. Look before you model

Before choosing what the classifier should be, find out what it has to swallow. The questions below
are the ones that turn out to matter for this particular model — answer them in the four cells that
follow, one cell per block, and keep the numbers, because the next sections read them back.

**A. Shape and types.**

1. `data.head()` and `data.dtypes`: which columns arrived as numbers and which as strings? Hold
   that against the `type` column of `variables` — do pandas and UCI agree everywhere?
2. `data.nunique()`, sorted, read as a ladder. Where does "a handful of labels" become "nearly one
   value per row"? Is the step as clean as you expected, and is anything stuck in the middle?

**B. The target, and the holes.**

3. `data[TARGET].value_counts()`. There are more than two distinct label strings — find out how
   many and why. (Adult ships as two files, and one of them punctuates.) Handed this column
   unprocessed, a classifier would be solving a different problem; which one? Fix the column in
   place — block D reads it back. Section 4 repeats the fix for the pipeline, so nothing later
   depends on your having done it here.
4. Once that is fixed, what share of the rows is the positive class? A model that always answers
   the majority scores exactly the complement. Write that number down — it is the floor everything
   later has to clear.
5. `data.isna().sum()`, then the number of rows carrying at least one gap. Three columns hold all
   of them. Express the loss as a percentage before deciding whether dropping those rows is cheap.

**C. The numeric columns.**

6. `data.describe()`. `fnlwgt` is a census sampling weight — look at its range. `capital-gain` —
   what share of the rows is *exactly* zero? Plot a histogram of each; neither looks like the bell
   curve a Gaussian would like to see.
7. Now plot `age` beside `education-num`. One is a measurement and the other is a ladder of labels
   that happens to be stored as integers. Which is which, and what in the plot gives it away?
   `pd.crosstab(data["education"], data["education-num"])` settles the question outright.

**D. Does any of it carry signal?**

8. Take one categorical column — `relationship` is a good one — and compute the positive rate per
   category against the base rate from question 4. Which categories move it furthest, and by how
   much?
9. For `age` and for `fnlwgt`, look at the distribution conditioned on the label:
   `data.groupby(TARGET)["age"].describe()`, or two overlaid histograms. One of the two separates
   the classes; the other does not separate them at all. Which is which?
10. Finally, a question about *pairs* of columns. Are `education` and `education-num` telling you
    two things, or one thing twice? What about `marital-status` and `relationship`? Keep the
    answer — section 6 comes back to it with a name.

In [ ]:
# A — shape and types.
# 1. `data.head()`, `data.dtypes`, against `variables["type"]`.
# 2. `data.nunique().sort_values()` — where is the step, and what sits in the middle?

# YOUR CODE HERE

In [ ]:
# B — the target, and the holes.
# 3. How many distinct label strings does `data[TARGET]` hold, and why more than two?
# 4. The positive share after cleaning, and the majority-class floor.
# 5. `data.isna().sum()`, the number of rows with at least one gap, and that as a percentage.

# YOUR CODE HERE

In [ ]:
# C — the numeric columns.
# 6. `data.describe()`; the range of `fnlwgt`; the zero share of `capital-gain`; a histogram of each.
# 7. `age` and `education-num` side by side, plus
#    `pd.crosstab(data["education"], data["education-num"])`.

# YOUR CODE HERE

In [ ]:
# D — signal, and overlap.
# 8. The positive rate per `relationship` category, against the base rate.
# 9. `age` and `fnlwgt` conditioned on the label — which one separates the classes?
# 10. `education` vs `education-num`, and `marital-status` vs `relationship`: two things or one?

# YOUR CODE HERE

### What the data turns out to be

Compare your answers against these — the rest of the notebook assumes them.

- **The label needed cleaning.** Adult ships as a train file and a test file, and the test file
  writes the class with a trailing period, so the raw column holds four strings for two classes:
  `<=50K` (24 720), `<=50K.` (12 435), `>50K` (7 841), `>50K.` (3 846). Strip the period and 24.2%
  of the rows are positive, which puts the majority-class floor at **75.8%** accuracy.
- **The gaps are concentrated and cheap.** `occupation` (966), `workclass` (963) and
  `native-country` (274) account for every one of them. 1 221 rows — 2.5% — carry at least one,
  leaving 47 621. Dropping them is what the contest does.
- **No single ladder sorts the columns into kinds.** Cardinality runs 2 (`sex`), 5, 6, 7, 9, 15, 16,
  16, 42 (`native-country`), 74 (`age`), 96, 99, 123, 28 523 (`fnlwgt`) — with no clean break: 42
  countries sit right next to 74 ages. Dtype settles it no better. `education-num` is an `int64` and
  so is `capital-gain`, and the two are not remotely the same kind of thing.
- **`education-num` is `education`, numbered.** The cross-tab is a 16 × 16 permutation matrix — a
  bijection. It is one variable recorded twice, once as a label and once as a rung.
- **Two numeric columns are badly behaved.** `fnlwgt` is a sampling weight spanning 12 285 to
  1 490 400, and its two class-conditional distributions are all but identical (means 190 186 and
  188 292): it says almost nothing about income. `capital-gain` is 91.7% exact zeros with a long
  thin tail — the signal it does carry is mostly *whether* it is zero (95.8% of the `<=50K` rows
  against 78.7% of the `>50K` ones), which is not a shape any bell curve has. Both come back in
  section 9.
- **Plenty of it does carry signal.** Mean age is 36.9 among `<=50K` against 44.2 among `>50K`;
  mean hours per week, 39.0 against 45.5. `relationship` swings the positive rate from 1.5%
  (`Own-child`) to 47.4% (`Wife`) around a 24.2% base. This is a learnable problem, not a coin
  flip.

## 3. Two kinds of feature, and what each one admits

Section 2 sorted the fourteen columns into two piles by eye. It pays to be precise about what the
piles are, because the model then falls out of the distinction almost mechanically.

### A categorical feature

`workclass` takes one of nine labels — `Private`, `Self-emp-not-inc`, `Federal-gov`, and so on. The
set is finite, the labels carry no order, and arithmetic on them means nothing: `Private` is not
twice `State-gov` and does not lie between two other categories. So the only thing the training
rows can tell you about such a column is how often each label occurred, and the only question you
can put to it at prediction time is *how often was this label seen in this class*. The estimate is
therefore a **table**: one number per (class, category) pair, $K_j$ categories wide.

Two awkward consequences come attached, and both show up in the code you are about to write. A cell
can come out empty — some label that never co-occurred with some class — and an empty cell means
probability zero, which inside a product annihilates every other feature's opinion. And a label can
turn up at prediction time for which the table has no column at all; it still has to be scored
somehow.

### A continuous feature

`age` is a measurement: ordered, with meaningful differences, and across 47 621 rows it takes 74
distinct values. `fnlwgt` takes 27 946. Here a table is not merely awkward, it is the wrong object.
Every entry would rest on a handful of rows, and a value absent from training is the common case
rather than a rare accident — so the table would answer "probability zero" to most of the test set.

What takes its place is a **density**: a single function on the whole real line, fitted to the
values seen within a class and evaluable at points it has never seen. A normal distribution with
that class's own mean and variance is the cheapest such function, and the default here. A kernel
density estimate is the obvious step up, and section 9 is about when the step is worth taking.

### And a third thing, dressed as the second

`education-num` is stored as `int64`, so every dtype check in pandas files it under numbers. Your
cross-tab showed what it actually is: a rung number for `education`, sixteen levels, in bijection
with a string column sitting a few places to its left. Arithmetic on it is *nearly* meaningful —
the rungs are genuinely ordered, and the positive rate broadly climbs along them, from 1.3% at
`Preschool` to 74.4% at `Prof-school` — but the spacing between rungs is invented. One rung from
`Bachelors` to `Masters` is not the same quantity of anything as one rung from `9th` to `10th`.

This is the case a column's storage type cannot decide, and it is why the next section reads types
off declared metadata rather than guessing from values.

### Why this model

A classifier that insists on one kind of feature has two ways out, and neither is good. Keep the
numbers and drop the labels, and `occupation`, `education` and `marital-status` — three of the
strongest columns in the table — go in the bin. Or coerce the labels into numbers, which asserts an
order and a spacing the data does not have, and hands a Gaussian the claim that `Private` = 3 sits
halfway between categories 2 and 4.

Naive Bayes needs neither, and that is the whole reason this assignment is this model. Its single
assumption — features independent once the class is known — turns the likelihood into a **product
of one factor per feature**, and nothing anywhere requires those factors to come from the same
family. A nine-cell table for `workclass`, a normal density for `age`, a kernel estimate for
`capital-gain`: multiply them together and the mixed record is handled as it comes, with no
encoding and no binning. Section 6 writes that product out, term by term.

## 4. Reading a column's kind off the metadata

The contest needs a rule rather than a judgement call, so it follows UCI's declared type with one
correction: a variable UCI calls `Integer` that takes at most ten distinct values is a code, not a
measurement. That is the `education-num` case from the last section, generalised.

On Adult the correction does not fire. `education-num` has sixteen levels, which is above the cut,
so the contest hands it to a Gaussian even though you and I know it is a ladder — the price of a
rule that has to run unattended, and a mild one, because the rungs are at least ordered. Where the
rule does earn its keep is a column that spells a four-level scale as `1..4`: declared `Integer`,
obviously a code, and silently disastrous as a measurement.

Rows with any missing value are dropped. The classifier you write is not asked to handle `NaN`, and
quietly imputing would change the numbers the reference was measured on.

In [ ]:
MAX_DISCRETE_CARDINALITY = 10

categorical_names, continuous_names = [], []
for column in adult.data.features.columns:
    declared = str(variables.loc[column, "type"])
    coded = declared == "Integer" and data[column].nunique() <= MAX_DISCRETE_CARDINALITY
    if declared in {"Continuous", "Integer"} and not coded:
        continuous_names.append(str(column))
    else:
        categorical_names.append(str(column))

clean = data.loc[data.notna().all(axis=1)].copy()
# The Adult test split spells the labels with a trailing period; normalise both forms.
clean[TARGET] = clean[TARGET].astype(str).str.strip().str.rstrip(".")
features = clean[categorical_names + continuous_names]
target = clean[TARGET]

print(f"categorical ({len(categorical_names)}): {categorical_names}")
print(f"continuous  ({len(continuous_names)}): {continuous_names}")
print(f"kept {len(clean)} of {len(data)} rows")
print(target.value_counts().to_string())

## 5. A split that does not move

`train_test_split` draws a random partition. Change the seed, the row order or the pandas version
and the partition changes with it — and so does every metric measured on it, which makes a
reference number meaningless. The contest therefore splits on a hash of **the row's own content**:

$$\text{row} \longmapsto \texttt{blake2b}(\text{row}) \bmod 10^4 < 2500 \implies \text{test}.$$

Three properties follow, and all three are why it is worth the extra code. The split is a pure
function of the data, so it is identical on every machine and every run. Duplicate rows always
land on the same side, so a repeated record cannot leak from train to test. And adding rows to the
dataset leaves the existing assignment untouched.

Class balance is only approximately preserved — this is not a stratified split — which is the one
price paid.

In [ ]:
categorical_values, continuous_values = mixed_nb_data.normalize_frame(
    features, categorical_names, continuous_names
)
labels = target.to_numpy(dtype=np.str_)

is_test = mixed_nb_data.hash_split(
    categorical_values, continuous_values, labels, test_fraction=0.25
)
is_train = ~is_test

print(f"train {is_train.sum()} / test {is_test.sum()} ({is_test.mean():.3%} on the test side)")
print(
    "re-running is a no-op:",
    np.array_equal(
        is_test,
        mixed_nb_data.hash_split(categorical_values, continuous_values, labels, 0.25),
    ),
)
print(
    "positive share, train vs test:",
    f"{(labels[is_train] == '>50K').mean():.4f}",
    f"{(labels[is_test] == '>50K').mean():.4f}",
)

In [ ]:
def build_matrix(regime: str) -> tuple[np.ndarray, list[int]]:
    """Returns the object feature matrix and the categorical column indices for one regime."""
    blocks = {
        "categorical": [categorical_values],
        "continuous": [continuous_values],
        "mixed": [categorical_values, continuous_values],
    }[regime]
    matrix = np.concatenate([block.astype(object) for block in blocks], axis=1)
    n_categorical = 0 if regime == "continuous" else len(categorical_names)
    return matrix, list(range(n_categorical))


def split(
    regime: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[int]]:
    """Returns `(X_train, X_test, y_train, y_test, categorical_indices)` for one regime."""
    matrix, categorical_indices = build_matrix(regime)
    return (
        matrix[is_train],
        matrix[is_test],
        labels[is_train],
        labels[is_test],
        categorical_indices,
    )


for regime in mixed_nb_data.REGIMES:
    X_train, X_test, _, _, categorical_indices = split(regime)
    print(
        f"{regime:11s} train {X_train.shape}  test {X_test.shape}  "
        f"categorical columns {len(categorical_indices)}"
    )

## 6. The model, term by term

What you are about to implement is Bayes' rule with one assumption bolted on. For a record
$x = (x_1, \dots, x_d)$ and a class $c$,

$$p(y = c \mid x) = \frac{p(y = c)\, p(x \mid y = c)}{p(x)},$$

and the *naive* assumption is that **once the class is known** the features are independent:

$$p(x \mid y = c) = \prod_{j=1}^{d} p(x_j \mid y = c).$$

Take logarithms — a product of fourteen likelihoods underflows to zero in double precision long
before you are finished — and split the sum by feature type:

$$\log p(y = c \mid x) = \underbrace{\log \pi_c}_{\text{prior}}
    + \underbrace{\sum_{j \in C} \log \hat p_{cj}(x_j)}_{\text{categorical}}
    + \underbrace{\sum_{j \in Q} \log \hat f_{cj}(x_j)}_{\text{continuous}}
    - \underbrace{\log Z(x)}_{\text{evidence}},$$

with $C$ the categorical features and $Q$ the continuous ones. Four terms, one at a time.

### $\log \pi_c$ — the prior

How common the class is before the record is read, estimated by its training frequency
$\hat\pi_c = N_c / N$. On Adult that is $0.758$ and $0.242$, which is to say the prior on its own
already answers `<=50K` to everything and is right three times in four — the floor from question 4.
Every other term's job is to move a record off that default. The `priors` parameter overrides the
frequencies: pass `[0.5, 0.5]` and you ask what the record says while ignoring how common the class
is, which is what you want when the training balance is an artefact of collection rather than a
fact about the world. `self._class_priors` already handles this one for you.

### $\log \hat p_{cj}(x_j)$ — one factor per categorical column

Count, then normalise within the class:

$$\hat p_{cj}(a) = \frac{N_{cja} + \alpha}{N_c + \alpha K_j},$$

where $N_{cja}$ is the number of training rows of class $c$ whose feature $j$ equals $a$, $N_c$ the
number of rows of class $c$, and $K_j$ the number of distinct categories of feature $j$ seen while
fitting. At $\alpha = 0$ this is the plain conditional frequency, and that is exactly what you must
not use: one empty cell — a category that never co-occurred with class $c$ in training — sends the
whole product to zero, $-\infty$ in logs, and no amount of evidence from the other thirteen columns
can pull it back. Adding $\alpha$ to every count puts a floor under every cell, at the price of a
slight pull toward uniform; $\alpha K_j$ in the denominator is what keeps the row summing to one.
This is Lidstone smoothing, and $\alpha = 1$ is the special case Laplace argued for.

A category met for the first time at **prediction** time has no column in the table whatsoever.
Score it as the zero-count case,

$$\hat p_{cj}(\text{unseen}) = \frac{\alpha}{N_c + \alpha K_j},$$

which is what `unseen_log_prob_` stores. Notice it is the same for every unseen value and varies
across classes only through $N_c$ — so a record whose categories are all novel falls back to very
nearly the prior, which is the right thing to say about a record you know nothing about.

### $\log \hat f_{cj}(x_j)$ — one factor per continuous column

A table is hopeless here, for the reasons section 3 gave, so estimate a **density** $\hat f_{cj}$:
a function on the whole line, fitted to the values of feature $j$ among the training rows of class
$c$ and no others. Two properties of densities trip people up, and both will show up in your
debugging:

- A density is not a probability. $\hat f(x)$ may exceed 1, and for a narrow peak it exceeds it by
  a lot, so a log-density above zero is normal rather than a bug.
- What is *missing* from the formula is the $\mathrm{d}x$ that would turn it into a probability. It
  is the same factor for every class, so it cancels in the normalisation below — which is precisely
  why adding a log-density and a log-probability-mass in one sum is legitimate. The posterior comes
  out right; the joint score on its own is right only up to that common factor.

The default $\hat f$ is `GaussianDensity`, a normal distribution fitted by maximum likelihood, so
$\hat\mu_{cj}$ and $\hat\sigma^2_{cj}$ are just the mean and the variance of that one column within
that one class, and

$$\log \hat f_{cj}(x) = -\tfrac{1}{2}\left[
    \log\!\left(2\pi\hat\sigma^2_{cj}\right) + \frac{(x - \hat\mu_{cj})^2}{\hat\sigma^2_{cj}}
\right].$$

The variance is clamped from below by `var_smoothing`, because a column that happens to be constant
within a class has $\hat\sigma^2 = 0$ and the expression would then be infinite everywhere else.
Nothing ties you to a Gaussian, though: `density_estimator` accepts any object with
`KernelDensity`'s `fit` / `score_samples` pair, and after section 2's look at `capital-gain` you can
guess why that matters.

### $-\log Z(x)$ — the evidence

The normaliser, $Z(x) = p(x) = \sum_{c'} \pi_{c'} \prod_j p(x_j \mid y = c')$, summed over classes.
It does not depend on $c$, so it shifts every class's score by the same amount and changes no
arg-max — `predict` could skip it entirely. What it buys is that the scores become a distribution
summing to one, which `predict_proba` and `log_loss` need. In log space it is a `logsumexp` across
the class axis, and **this term is already written**: `predict_log_proba` subtracts it from whatever
`_joint_log_likelihood` returns. Your work stops at the first three terms.

### How wrong is "naive"?

Badly wrong, here — and that is worth knowing before the numbers surprise you. Question 10 found
`education` and `education-num` to be one variable twice over, and `marital-status` and
`relationship` to overlap almost completely (`Married-civ-spouse` is `Husband` or `Wife` in all but
a few hundred rows). Each duplicated pair gets its evidence counted twice, and the posteriors come
out over-confident, pushed toward 0 and 1 harder than the data warrants. The arg-max largely
survives it, which is why naive Bayes stays a respectable classifier while being a poor probability
estimate — and why the contest scores accuracy and macro $F_1$ rather than calibration.

## 7. What you implement

Four gaps in `naive_bayes.py`, in the order they are easiest to get right:

1. `GaussianDensity.fit` — $\hat\mu$ and $\hat\sigma^2$ by maximum likelihood, the variance clamped
   from below by `var_smoothing`.
2. `GaussianDensity.score_samples` — the log-density written out in the third term above, evaluated
   at every query point.
3. `MixedNaiveBayes.fit` — the priors, the smoothed categorical tables, and one cloned density
   estimator per class and per continuous feature.
4. `MixedNaiveBayes._joint_log_likelihood` — $\log \pi_c$ plus one term per feature. The fourth
   term is not yours.

Two traps worth naming in advance. Work in log space throughout: a product of fourteen likelihoods
underflows to zero in double precision, and `predict_proba` only looks fine because
`predict_log_proba` normalises with `logsumexp`. And clone the density estimator with
`sklearn.base.clone` before fitting it — the caller passes **one** object for all classes and
features, and fitting it in place would leave every class sharing the last fit.

Everything that is not the model — column plumbing, the posterior normalisation, parameter
validation — is already written, and the cell below reloads the module so you can edit the file and
re-run without restarting the kernel.

In [ ]:
import naive_bayes

importlib.reload(naive_bayes)
GaussianDensity = naive_bayes.GaussianDensity
MixedNaiveBayes = naive_bayes.MixedNaiveBayes

X_train, X_test, y_train, y_test, categorical_indices = split("mixed")
model = MixedNaiveBayes(categorical_features=categorical_indices).fit(X_train, y_train)
predicted = model.predict(X_test)

print(f"classes            {model.classes_.tolist()}")
print(f"priors             {np.exp(model.class_log_prior_)}")
print(
    f"categorical / cont {model.categorical_features_.tolist()} / "
    f"{model.continuous_features_.tolist()}"
)
print(f"accuracy           {accuracy_score(y_test, predicted):.4f}")
print(f"macro F1           {f1_score(y_test, predicted, average='macro'):.4f}")
log_loss_value = log_loss(y_test, model.predict_proba(X_test), labels=model.classes_)
print(f"log loss           {log_loss_value:.4f}")

### Reference numbers

What the reference implementation reaches on this dataset with the defaults — `alpha=1.0`,
`GaussianDensity`, the 75/25 hash split, 35 713 train and 11 908 test rows once the 1 221 incomplete
ones are dropped:

| regime | accuracy | macro $F_1$ | log loss |
| --- | --- | --- | --- |
| categorical only | 0.7956 | 0.7463 | 0.4573 |
| continuous only | 0.7995 | 0.6517 | 0.9065 |
| full mixed table | 0.8332 | 0.7448 | 0.6659 |

Land within about 0.02 of the accuracy column and your implementation agrees with the one that
grades you. Landing well *above* it is not good news: it almost always means the test rows leaked
into training.

One thing not to read into the last row: the mixed table wins here, but it does not have to. A
continuous column the Gaussian models badly — `capital-gain`, 91.7% zeros with a long tail, is the
one you have already met — can pull the mixture *below* the categorical half. A drop when you add
the continuous columns is a result, not necessarily a bug; the fix is a better density, which is
what `density_estimator` is for.

## 8. Baselines

Two of these are not just baselines but equivalences, and they are the sharpest debugging tool you
have:

- on continuous features only with the default `GaussianDensity`, the model **is**
  `GaussianNB(var_smoothing=0)` — posteriors should agree to machine precision;
- on categorical features only, it **is** `CategoricalNB(alpha=1)` on ordinal-encoded columns —
  likewise.

If either disagrees, the bug is in that one likelihood, which is a much smaller place to look than
"the classifier is off".

In [ ]:
rows = []

X_train_c, X_test_c, y_train_c, y_test_c, _ = split("continuous")
X_train_c = X_train_c.astype(np.float64)
X_test_c = X_test_c.astype(np.float64)
mine = MixedNaiveBayes(categorical_features=[]).fit(X_train_c, y_train_c)
theirs = GaussianNB(var_smoothing=0.0).fit(X_train_c, y_train_c)
rows.append(
    (
        "continuous: mine vs GaussianNB",
        accuracy_score(y_test_c, mine.predict(X_test_c)),
        accuracy_score(y_test_c, theirs.predict(X_test_c)),
        np.abs(mine.predict_log_proba(X_test_c) - theirs.predict_log_proba(X_test_c)).max(),
    )
)

X_train_k, X_test_k, y_train_k, y_test_k, categorical_k = split("categorical")
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(X_train_k)
mine_k = MixedNaiveBayes(categorical_features=categorical_k, alpha=1.0).fit(X_train_k, y_train_k)
theirs_k = CategoricalNB(alpha=1.0, force_alpha=True).fit(encoder.transform(X_train_k), y_train_k)
rows.append(
    (
        "categorical: mine vs CategoricalNB",
        accuracy_score(y_test_k, mine_k.predict(X_test_k)),
        accuracy_score(y_test_k, theirs_k.predict(encoder.transform(X_test_k))),
        np.abs(
            mine_k.predict_log_proba(X_test_k)
            - theirs_k.predict_log_proba(encoder.transform(X_test_k))
        ).max(),
    )
)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
rows.append(
    (
        "mixed: mine vs majority class",
        accuracy_score(y_test, model.predict(X_test)),
        accuracy_score(y_test, dummy.predict(X_test)),
        float("nan"),
    )
)

display(pd.DataFrame(rows, columns=["comparison", "mine", "reference", "max |Δ log p|"]))

## 9. Swapping the continuous likelihood

`density_estimator` is the point of the exercise that is not naive Bayes: anything with
`KernelDensity`'s `fit` / `score_samples` pair can model a continuous feature, and the Gaussian is
only the default because it is the cheapest honest choice.

Two things to keep in mind when you reach for a real KDE.

**Scale.** `KernelDensity(bandwidth="scott")` sets $h = n^{-1/(d+4)}$ and does *not* multiply by
the feature's standard deviation. On `capital-gain`, which runs to $10^5$, a bandwidth of $0.25$
turns the density into a comb of spikes. Standardise the continuous block first, or pass a
bandwidth in the feature's own units.

**Cost.** Scoring a KDE is $O(n_\text{train} \cdot n_\text{test})$ per class and per feature,
against $O(n_\text{train} + n_\text{test})$ for the Gaussian. On the full Adult that is billions of
kernel evaluations, so the comparison below runs on a deterministic subsample — carved out with the
same hash trick, under a different salt, so it too is reproducible.

In [ ]:
subsample = mixed_nb_data.hash_split(
    categorical_values,
    continuous_values,
    labels,
    test_fraction=0.03,
    salt="kde-demo",
)
sub_train, sub_test = subsample & is_train, subsample & is_test
scaler = StandardScaler().fit(continuous_values[sub_train])
X_sub_train = scaler.transform(continuous_values[sub_train])
X_sub_test = scaler.transform(continuous_values[sub_test])
y_sub_train, y_sub_test = labels[sub_train], labels[sub_test]
print(f"subsample: {len(y_sub_train)} train / {len(y_sub_test)} test")

rows = []
candidates = {
    "GaussianDensity (default)": None,
    "KDE gaussian h=0.2": KernelDensity(kernel="gaussian", bandwidth=0.2),
    "KDE gaussian h=0.5": KernelDensity(kernel="gaussian", bandwidth=0.5),
    "KDE epanechnikov h=0.5": KernelDensity(kernel="epanechnikov", bandwidth=0.5),
    "KDE tophat h=0.8": KernelDensity(kernel="tophat", bandwidth=0.8),
}
for name, estimator in candidates.items():
    started = time.perf_counter()
    candidate = MixedNaiveBayes(categorical_features=[], density_estimator=estimator).fit(
        X_sub_train, y_sub_train
    )
    fitted_at = time.perf_counter()
    guess = candidate.predict(X_sub_test)
    rows.append(
        (
            name,
            accuracy_score(y_sub_test, guess),
            f1_score(y_sub_test, guess, average="macro"),
            (fitted_at - started) * 1e3,
            (time.perf_counter() - fitted_at) * 1e3,
        )
    )

display(pd.DataFrame(rows, columns=["density", "accuracy", "macro F1", "fit, ms", "predict, ms"]))

## 10. Where the time goes

The contest measures three stages — constructing the estimator, fitting it, predicting — and
compares each against the reference solution's time with a wide factor plus a floor. Wide enough
for a slow host; not wide enough for a per-row Python loop that refits a density inside `predict`.
`mixed_nb_eval.evaluate` is the very function the grader uses, so this table is the one it will
produce.

In [ ]:
timings = []
for regime in mixed_nb_data.REGIMES:
    X_tr, X_te, y_tr, y_te, categorical_indices = split(regime)
    evaluation = mixed_nb_eval.evaluate(
        lambda indices=categorical_indices: MixedNaiveBayes(categorical_features=indices),
        X_tr,
        y_tr,
        X_te,
        y_te,
    )
    timings.append(
        {
            "regime": regime,
            "accuracy": evaluation.accuracy,
            "macro F1": evaluation.macro_f1,
            "log loss": evaluation.log_loss,
            **{
                f"{stage}, ms": evaluation.runtime_seconds[stage] * 1e3
                for stage in mixed_nb_eval.STAGES
            },
        }
    )

display(pd.DataFrame(timings).set_index("regime"))

## 11. Local tests

Six checks, each standing for one group of the grader's tests. They run on this dataset rather than
on the grader's, so passing them all is necessary and not sufficient — but every one of them fails
for a specific, nameable reason, which is what makes them worth running before you submit.

In [ ]:
# Test 1 — the sklearn estimator contract.
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.pipeline import Pipeline

fresh = MixedNaiveBayes(categorical_features=categorical_indices)
assert set(fresh.get_params()) == {
    "categorical_features",
    "density_estimator",
    "alpha",
    "priors",
}, "keep the constructor signature as given"
assert clone(fresh).get_params()["alpha"] == 1.0, "clone must preserve the parameters"
try:
    clone(fresh).predict(X_test)
except NotFittedError:
    pass
else:
    raise AssertionError("predict before fit must raise NotFittedError")

X_tr, X_te, y_tr, y_te, indices = split("mixed")
assert MixedNaiveBayes(categorical_features=indices).fit(X_tr, y_tr) is not None
scores = cross_val_score(
    MixedNaiveBayes(categorical_features=indices), X_tr[:3000], y_tr[:3000], cv=4
)
assert np.all(np.isfinite(scores)), f"cross_val_score produced {scores}"
assert (
    Pipeline([("nb", MixedNaiveBayes(categorical_features=indices))])
    .fit(X_tr, y_tr)
    .score(X_te, y_te)
    > 0.6
), "the estimator must work as a Pipeline step"
print(f"OK — estimator contract holds, 4-fold CV accuracy {scores.mean():.4f}")

In [ ]:
# Test 2 — categorical features only: the model is CategoricalNB.
X_tr, X_te, y_tr, y_te, indices = split("categorical")
mine = MixedNaiveBayes(categorical_features=indices, alpha=1.0).fit(X_tr, y_tr)
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(X_tr)
theirs = CategoricalNB(alpha=1.0, force_alpha=True).fit(encoder.transform(X_tr), y_tr)
gap = np.abs(mine.predict_log_proba(X_te) - theirs.predict_log_proba(encoder.transform(X_te))).max()
assert gap < 1e-6, f"posteriors differ from CategoricalNB by {gap:.3e}"
print(f"OK — matches CategoricalNB to {gap:.2e}; accuracy {mine.score(X_te, y_te):.4f}")

In [ ]:
# Test 3 — continuous features only: the model is GaussianNB.
X_tr, X_te, y_tr, y_te, _ = split("continuous")
X_tr, X_te = X_tr.astype(np.float64), X_te.astype(np.float64)
# Adult's `fnlwgt` has a variance of order 1e10, so the two log-densities cancel down to
# roughly 1e-9 absolute even when both are right; 1e-6 is still far tighter than any bug.
mine = MixedNaiveBayes(categorical_features=[]).fit(X_tr, y_tr)
theirs = GaussianNB(var_smoothing=0.0).fit(X_tr, y_tr)
gap = np.abs(mine.predict_log_proba(X_te) - theirs.predict_log_proba(X_te)).max()
assert gap < 1e-6, f"posteriors differ from GaussianNB by {gap:.3e}"
print(f"OK — matches GaussianNB to {gap:.2e}; accuracy {mine.score(X_te, y_te):.4f}")

In [ ]:
# Test 4 — the mixed table: probabilities are consistent and the mixture beats either half.
X_tr, X_te, y_tr, y_te, indices = split("mixed")
mixed = MixedNaiveBayes(categorical_features=indices).fit(X_tr, y_tr)
report = mixed_nb_eval.probability_report(mixed, X_te)
assert report["row_sum_deviation"] < 1e-9, "predict_proba rows must sum to one"
assert report["log_gap"] < 1e-9, "predict_log_proba must be log(predict_proba)"
assert report["argmax_disagreement"] == 0.0, "predict must be the arg-max of predict_proba"

mixed_accuracy = mixed.score(X_te, y_te)
majority = (y_te == pd.Series(y_tr).mode()[0]).mean()
assert mixed_accuracy > majority, (
    f"mixed accuracy {mixed_accuracy:.4f} does not beat the majority class {majority:.4f}"
)
print(
    f"OK — probabilities consistent; mixed accuracy {mixed_accuracy:.4f} vs majority {majority:.4f}"
)

In [ ]:
# Test 5 — a caller-supplied density estimator is used, and cloned rather than fitted in place.
from scipy.special import logsumexp
from sklearn.utils.validation import check_is_fitted

estimator = KernelDensity(kernel="gaussian", bandwidth=0.5)
model_kde = MixedNaiveBayes(categorical_features=[], density_estimator=estimator).fit(
    X_sub_train, y_sub_train
)
try:
    check_is_fitted(estimator)
except NotFittedError:
    pass
else:
    raise AssertionError("the estimator passed in must be cloned, not fitted in place")

classes, encoded = np.unique(y_sub_train, return_inverse=True)
expected = np.tile(np.log(np.bincount(encoded) / encoded.shape[0]), (len(y_sub_test), 1))
for position in range(X_sub_train.shape[1]):
    for class_index in range(len(classes)):
        density = KernelDensity(kernel="gaussian", bandwidth=0.5).fit(
            X_sub_train[encoded == class_index, position].reshape(-1, 1)
        )
        expected[:, class_index] += density.score_samples(X_sub_test[:, position].reshape(-1, 1))
expected -= logsumexp(expected, axis=1, keepdims=True)
gap = np.abs(model_kde.predict_log_proba(X_sub_test) - expected).max()
assert gap < 1e-8, f"the KDE posterior is off by {gap:.3e}"
print(f"OK — KernelDensity plugged in correctly, posterior matches to {gap:.2e}")

In [ ]:
# Test 6 — smoothing, unseen categories and explicit priors.
alpha = 0.3
X_tr, X_te, y_tr, y_te, indices = split("categorical")
smoothed = MixedNaiveBayes(categorical_features=indices, alpha=alpha).fit(X_tr, y_tr)
_, encoded = np.unique(y_tr, return_inverse=True)
class_counts = np.bincount(encoded).astype(np.float64)

column = X_tr[:, 0].astype(str)
categories = smoothed.categories_[0]
assert list(categories) == list(np.unique(column)), "categories_ must be the sorted categories"
for code_index, category in enumerate(categories):
    for class_index in range(len(class_counts)):
        joint = np.sum((column == category) & (encoded == class_index))
        expected_probability = (joint + alpha) / (
            class_counts[class_index] + alpha * len(categories)
        )
        assert np.isclose(
            np.exp(smoothed.feature_log_prob_[0][class_index, code_index]),
            expected_probability,
        ), f"the smoothed frequency for {category!r} is wrong"
assert np.allclose(
    np.exp(smoothed.unseen_log_prob_[:, 0]),
    alpha / (class_counts + alpha * len(categories)),
), "an unseen category must be scored as a zero-count one"

novel = np.array([["never-seen"] * len(indices)], dtype=object)
expected_row = smoothed.class_log_prior_ + smoothed.unseen_log_prob_.sum(axis=1)
expected_row -= logsumexp(expected_row)
assert np.allclose(smoothed.predict_log_proba(novel)[0], expected_row), (
    "a row of unseen categories must be scored with unseen_log_prob_"
)

uniform = MixedNaiveBayes(categorical_features=indices, priors=[0.5, 0.5]).fit(X_tr, y_tr)
assert np.allclose(uniform.class_log_prior_, np.log(0.5)), "priors must override the frequencies"
print("OK — Lidstone smoothing, unseen categories and explicit priors all behave")

## 12. Submitting

Upload **`naive_bayes.py`** — the file itself, not this notebook and not a zip. The grader renames
it, imports `MixedNaiveBayes` and `GaussianDensity` from it and runs its own tests; nothing else in
the file is executed, so leftover experiments in a `__main__` block are harmless but pointless.

Two differences from what you have been running here are worth expecting. Your classifier is scored
on datasets other than this one, so a bug that Adult's size and category counts gave room to hide in
has fewer places to go — which is also why the local checks above are necessary and not sufficient.
And the metrics are compared against a reference implementation two-sided: scoring far *above* the
reference means you are not fitting the stated model either.